# PF-5039: Análisis de Imágenes Médicas y Biológicas
## Semana 02 — Práctica de Laboratorio (1 Hora en Clase)
### Transformaciones de Intensidad y Análisis de Histograma

**Estudiante:** ____________________________________ **Carné:** __________________

---
### Objetivos de la Práctica:
1. Comprender la relación matemática entre el histograma empírico de frecuencias $h(r_k)$, la función de probabilidad normalizada (PMF) $p(r_k)$ y la **Entropía de Shannon** $H$.
2. Implementar y comparar transformaciones de intensidad puntuales: **Transformación Logarítmica** y **Corrección Gamma**.
3. Comparar la **Ecualización Global de Histograma (HE)** con la **Ecualización Adaptativa Limitada por Contraste (CLAHE)** sobre imágenes biomédicas con iluminación heterogénea.
4. Medir el impacto de la eficiencia computacional comparando bucles tradicionales píxel a píxel contra tablas de búsqueda (**Look-Up Tables / LUT**).

**Referencia:** González & Woods (2018), *Digital Image Processing*, 4th Ed., Cap. 3.

---
### Ejercicio 1: Inspección de Imagen, Histograma y Entropía de Shannon

**Contexto:** La Entropía de Shannon cuantifica el contenido promedio de información o incertidumbre en la distribución de intensidades:
$$H = -\sum_{k=0}^{L-1} p(r_k) \log_2\bigl(p(r_k)\bigr) \quad [\text{bits/píxel}]$$
donde $p(r_k) = \frac{h(r_k)}{M \cdot N}$ es la frecuencia relativa normalizada (PMF). Para intensidades ausentes ($p=0$), por definición estocástica $0 \log_2(0) \equiv 0$.

**Instrucciones:**
1. Cargue la imagen de microscopía `tejido_renal_fluorescencia.png` en escala de grises.
2. Imprima las dimensiones $(M, N)$, tipo de dato, valor mínimo, máximo y media.
3. Calcule la PMF normalizada $p(r_k)$ con `np.histogram` o `np.bincount`.
4. **Complete el código** para calcular la Entropía de Shannon $H$, ignorando los valores $p=0$.
5. Grafique la imagen junto con su histograma en un panel de 2 columnas.

In [1]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt

# Función auxiliar de carga compatible con rutas con acentos en Windows
def leer_imagen(ruta, flags=cv2.IMREAD_GRAYSCALE):
    img = cv2.imread(ruta, flags)
    if img is None and os.path.exists(ruta):
        img = cv2.imdecode(np.fromfile(ruta, dtype=np.uint8), flags)
    return img

# 1. Cargar imagen en escala de grises
img_renal = leer_imagen('../../Imagenes_Ejemplo/tejido_renal_fluorescencia.png')
assert img_renal is not None, "Error al cargar tejido_renal_fluorescencia.png"

print(f"Dimensiones: {img_renal.shape} | Tipo: {img_renal.dtype}")
print(f"Mínimo: {img_renal.min()} | Máximo: {img_renal.max()} | Media: {img_renal.mean():.2f}")

# 2. Calcular histograma y probabilidad normalizada (PMF)
hist, bins = np.histogram(img_renal.ravel(), bins=256, range=[0, 256])
p = hist / hist.sum()

# ==========================================================================
# TODO: Calcule la Entropía de Shannon H en bits/píxel
# Pista: filtre p > 0 antes de aplicar np.log2 para evitar log(0)
# ==========================================================================
# H = ...
H = 0.0  # <-- Reemplazar por su implementación

# 3. Visualización
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].imshow(img_renal, cmap='gray')
axes[0].set_title('Tejido Renal (Fluorescencia)', fontweight='bold')
axes[0].axis('off')

axes[1].bar(bins[:-1], hist, width=1.0, color='#1F4E79', alpha=0.85)
axes[1].set_title(f'Histograma de Intensidad — Entropía H = {H:.3f} bits/px', fontweight='bold')
axes[1].set_xlabel('Nivel de Gris (r)')
axes[1].set_ylabel('Frecuencia de Píxeles')
axes[1].set_xlim([0, 255])
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

[ WARN:0@0.198] global loadsave.cpp:278 findDecoder imread_('../../Imagenes_Ejemplo/tejido_renal_fluorescencia.png'): can't open/read file: check file path/integrity


AssertionError: Error al cargar tejido_renal_fluorescencia.png

---
### Ejercicio 2: Transformaciones de Intensidad (Logarítmica y Ley de Potencias / Gamma)

**Contexto:**
- **Transformación Logarítmica:** $s = c \cdot \log(1 + r)$, donde la constante de escala es $c = \frac{255}{\log(1 + r_{\max})}$. Expande el rango dinámico de las intensidades bajas y comprime los brillos altos.
- **Corrección Gamma (Ley de Potencias):** $s = c \cdot r^\gamma$. Normalizando a $[0, 1]$, se calcula como: $s = 255 \cdot \left(\frac{r}{255}\right)^\gamma$.
  - $\gamma < 1$: Aclara regiones oscuras y expande contraste en sombras.
  - $\gamma > 1$: Oscurece la imagen y expande contraste en regiones claras.

**Instrucciones:**
1. Cargue la imagen `tomografia_expansion_low.png`.
2. **Complete el código** de las funciones `transformacion_logaritmica(img)` y `transformacion_gamma(img, gamma)`.
3. Genere los resultados para transformación logarítmica, $\gamma = 0.5$ y $\gamma = 2.2$.
4. Compare visualmente las imágenes transformadas y analice cómo cambian sus histogramas.

In [ ]:
# 1. Cargar imagen de prueba
img_tomo = leer_imagen('../../Imagenes_Ejemplo/tomografia_expansion_low.png')
assert img_tomo is not None, "Error al cargar tomografia_expansion_low.png"

def transformacion_logaritmica(img):
    """
    Aplica la transformación s = c * log(1 + r) normalizada a uint8 [0, 255].
    """
    # ======================================================================
# TODO: Implementar transformación logarítmica
    # ======================================================================
    # c = 255.0 / np.log(1.0 + np.max(img))
    # s = c * np.log(1.0 + img.astype(np.float64))
    # return np.clip(s, 0, 255).astype(np.uint8)
    return img.copy()  # <-- Reemplazar por su código

def transformacion_gamma(img, gamma):
    """
    Aplica corrección gamma s = 255 * (r / 255)^gamma.
    """
    # ======================================================================
# TODO: Implementar transformación gamma
    # ======================================================================
    # s = 255.0 * np.power(img.astype(np.float64) / 255.0, gamma)
    # return np.clip(s, 0, 255).astype(np.uint8)
    return img.copy()  # <-- Reemplazar por su código

# 2. Ejecutar transformaciones
img_log = transformacion_logaritmica(img_tomo)
img_gamma_bright = transformacion_gamma(img_tomo, gamma=0.5)  # Aclara
img_gamma_dark = transformacion_gamma(img_tomo, gamma=2.2)    # Oscurece

# 3. Visualización comparativa
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
titulos = ['Original (Bajo Contraste)', 'Transformación Logarítmica', 'Gamma = 0.5 (Aclarar)', 'Gamma = 2.2 (Oscurecer)']
imagenes = [img_tomo, img_log, img_gamma_bright, img_gamma_dark]

for i in range(4):
    axes[0, i].imshow(imagenes[i], cmap='gray')
    axes[0, i].set_title(titulos[i], fontweight='bold', fontsize=10)
    axes[0, i].axis('off')
    
    axes[1, i].hist(imagenes[i].ravel(), bins=64, range=[0, 256], color='#1F4E79', alpha=0.75)
    axes[1, i].set_xlim([0, 255])
    axes[1, i].set_xlabel('Intensidad')
    axes[1, i].grid(alpha=0.3)

axes[1, 0].set_ylabel('Frecuencia')
plt.tight_layout()
plt.show()

---
### Ejercicio 3: Ecualización Global (HE) vs Ecualización Adaptativa (CLAHE)

**Contexto:**
- **HE (Histogram Equalization):** Aplica una única CDF global para aplanar el histograma. En imágenes médicas con regiones homogéneas amplias (fondos oscuros o líquidos), amplifica el ruido térmico de forma destructiva.
- **CLAHE (Contrast Limited Adaptive Histogram Equalization):** Divide la imagen en bloques contextuales (grilla $8 \times 8$), recorta los picos locales con un umbral (*Clip Limit*) redistribuyendo el área uniformemente, e interpola bilinealmente las funciones de mapeo.

**Instrucciones:**
1. Cargue la radiografía médica de bajo contraste `rayos_x_bajo_contraste.jpeg` (o `ahe_demo_orig.png`).
2. Aplique la ecualización global `cv2.equalizeHist(img)`.
3. Cree y aplique un objeto CLAHE con `cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))`.
4. Calcule la Entropía de Shannon $H$ para cada una de las 3 imágenes.
5. Muestre los resultados en un panel comparativo de 3 columnas.

In [ ]:
# 1. Cargar radiografía de prueba
img_rx = leer_imagen('../../Imagenes_Ejemplo/rayos_x_bajo_contraste.jpeg')
if img_rx is None:
    img_rx = leer_imagen('../../Imagenes_Ejemplo/ahe_demo_orig.png')
assert img_rx is not None, "Error al cargar imagen radiológica"

# ==========================================================================
# TODO: Aplicar HE y CLAHE
# ==========================================================================
# img_he = cv2.equalizeHist(img_rx)
# clahe_obj = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
# img_clahe = clahe_obj.apply(img_rx)

img_he = img_rx.copy()      # <-- Reemplazar por cv2.equalizeHist
img_clahe = img_rx.copy()   # <-- Reemplazar por CLAHE

# Función auxiliar para calcular entropía
def calcular_entropia(im):
    h, _ = np.histogram(im.ravel(), bins=256, range=[0, 256])
    pk = h / h.sum()
    pk = pk[pk > 0]
    return -np.sum(pk * np.log2(pk))

h_orig = calcular_entropia(img_rx)
h_he = calcular_entropia(img_he)
h_clahe = calcular_entropia(img_clahe)

# 2. Visualización comparativa
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
imgs = [img_rx, img_he, img_clahe]
titles = [f'Original (H = {h_orig:.2f} bits)', f'HE Global (H = {h_he:.2f} bits)', f'CLAHE (H = {h_clahe:.2f} bits)']

for i in range(3):
    axes[0, i].imshow(imgs[i], cmap='gray')
    axes[0, i].set_title(titles[i], fontweight='bold')
    axes[0, i].axis('off')
    
    axes[1, i].hist(imgs[i].ravel(), bins=64, range=[0, 256], color='#C00000' if i==1 else '#1F4E79', alpha=0.75)
    axes[1, i].set_xlim([0, 255])
    axes[1, i].set_xlabel('Intensidad')
    axes[1, i].grid(alpha=0.3)

axes[1, 0].set_ylabel('Frecuencia')
plt.tight_layout()
plt.show()

---
### Ejercicio 4: Desempeño Computacional (Bucle Píxel a Píxel vs Look-Up Table / LUT)

**Contexto:** Las transformaciones puntuales no dependen de los vecinos espaciales. Para una imagen de $M \times N$ de 8 bits ($L=256$), ejecutar operaciones matemáticas complejas (como `np.power` o `np.log`) píxel por píxel requiere $M \cdot N$ evaluaciones. En contraste, una **Look-Up Table (LUT)** pre-calcula los 256 posibles resultados y mapea la imagen por indexación directa en memoria en tiempo $O(1)$ por píxel.

**Instrucciones:**
1. Cargue la imagen grande `gigante.png` (o genere una imagen de prueba de $2000 \times 2000$).
2. Mida el tiempo de ejecución promedio tras 5 repeticiones de una transformación (inversión o corrección gamma) mediante:
   - **Método 1:** Recorrer la imagen con bucles `for` anidados (píxel a píxel).
   - **Método 2:** Mapeo mediante Look-Up Table con `cv2.LUT()`.
3. Calcule el factor de aceleración (*Speedup*): $\text{Speedup} = \frac{t_{\text{pixel}}}{t_{\text{LUT}}}$.
4. Analice la relevancia de este resultado para el procesamiento de video o secuencias médicas en tiempo real.

In [ ]:
import time

# 1. Cargar imagen de prueba de alta resolución
img_gigante = leer_imagen('../../Imagenes_Ejemplo/gigante.png')
if img_gigante is None:
    # Si no existe, crear imagen sintética de 2000 x 2000
    img_gigante = np.random.randint(0, 256, size=(2000, 2000), dtype=np.uint8)

print(f"Dimensiones de la imagen para benchmark: {img_gigante.shape} ({img_gigante.size / 1e6:.2f} Megapíxeles)")

gamma = 0.6
iteraciones = 5

# ==========================================================================
# TODO: Método 1 - Recorrido manual píxel a píxel (para evaluar tiempo)
# ==========================================================================
H_sub, W_sub = 500, 500  # Sub-bloque para el bucle manual (250 mil píxeles)
sub_img = img_gigante[:H_sub, :W_sub]

t0 = time.perf_counter()
res_pixel = np.zeros_like(sub_img)
for y in range(H_sub):
    for x in range(W_sub):
        res_pixel[y, x] = int(255.0 * ((sub_img[y, x] / 255.0) ** gamma))
t_pixel_sub = (time.perf_counter() - t0)
# Extrapolar al tamaño completo de la imagen
t_pixel_estimado = t_pixel_sub * (img_gigante.size / sub_img.size)

# ==========================================================================
# TODO: Método 2 - Mapeo vectorial con Look-Up Table (cv2.LUT)
# ==========================================================================
# 1. Pre-calcular la tabla LUT de 256 valores uint8
lut_table = np.array([int(255.0 * ((i / 255.0) ** gamma)) for i in range(256)], dtype=np.uint8)

# 2. Medir tiempo de cv2.LUT sobre la imagen COMPLETA
t0 = time.perf_counter()
for _ in range(iteraciones):
    res_lut = cv2.LUT(img_gigante, lut_table)
t_lut = (time.perf_counter() - t0) / iteraciones

print(f"\n--- Resultados del Benchmark Computacional ---")
print(f"Tiempo estimado bucles for (Píxel a Píxel): {t_pixel_estimado:.4f} s")
print(f"Tiempo promedio cv2.LUT (Look-Up Table):    {t_lut:.6f} s")
print(f"Factor de Aceleración (Speedup):           {t_pixel_estimado / t_lut:.1f}x más rápido")